# 🧠 SynthRAD2023: Model 4 — Cross-Fitted ResUNet Residual Conditional Diffusion
## Two-Stage High-Fidelity Brain MRI-to-Synthetic CT Synthesis
### Scientific Research Paper: Advanced Synthesis Paradigm

This notebook implements **Model 4: Cross-Fitted ResUNet Residual Conditional Diffusion** for synthetic CT (sCT) generation from brain MRI on the **SynthRAD2023 Task 1 Brain cohort (UMC Utrecht)**.

---

### 🔬 Research Hypothesis:
Direct end-to-end regression (Model 1 ResUNet) rapidly captures macroscopic anatomical geometry and soft-tissue boundaries, but suffers from oversmoothing in high-frequency bony structures. Standard unconditional or full-image diffusion (Model 3 cDDPM) models the entire complex CT distribution from scratch, which requires substantial capacity and sampling steps.

**The Model 4 Hypothesis:**
Decoupling the synthesis task into:
1. **Stage 1 (Coarse Anatomical CT Estimation):** A supervised ResUNet predicting coarse anatomical CT $c = G_\phi(\text{MRI})$.
2. **Stage 2 (Cross-Fitted Residual Refinement Diffusion):** A conditional diffusion model learning solely the residual error $r = y - c_{\text{OOF}}$, conditioned on $[z_t, \text{MRI}, c, \text{mask}]$,
allows the diffusion denoiser to focus all modeling capacity on high-frequency bone refinement, structural fine-tuning, and residual error correction without having to reconstruct macroscopic anatomy from zero.

---

### 🛡️ Critical Scientific Protocol Standards:
- **Zero In-Sample Leakage via 4-Fold Cross-Fitting:** The residual diffusion model is trained **strictly on Out-Of-Fold (OOF) coarse predictions**. Each training patient's coarse prediction is generated by a ResUNet instance trained on the remaining 3 folds (never having seen that patient during training).
- **Ensemble Validation & Inference:** For validation and test patients, the coarse estimate is formed by the deterministic ensemble of all 4 fold models: $c = \frac{1}{4}\sum_{k=1}^4 G_{\phi_k}(\text{MRI})$, minimizing the training-inference distribution shift.
- **Fixed Residual Scale $s$:** Estimated exclusively from the training OOF residuals: $z_0 = r / s$. Normalized residuals $z_0$ are modeled without artificial $[-1, 1]$ clean-CT clipping.
- **Fixed Custom 60-Patient Subset:** 48 Train (2,879 slices), 6 Val (372 slices), 6 Test (363 slices) locked to `SEED = 42`.
  *(Note: This is a 2D PNG-based adaptation with a custom patient split, not the official 3D SynthRAD challenge split).*
- **Frozen Test Isolation:** Test patients remain strictly untouched during training, validation, cross-fitting, residual scaling, and checkpoint selection.
- **Isolated Workspace:** All artifacts and checkpoints are saved exclusively under `/kaggle/working/synthrad_model4_residual_diffusion/`.


In [ ]:
# =====================================================================
# CELL 1: ENVIRONMENT SETUP, HARDWARE INSPECTION & DETERMINISTIC SEEDING
# =====================================================================

import os
import sys
import glob
import math
import time
import json
import shutil
import random
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from skimage.metrics import structural_similarity as compute_ssim
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from scipy.spatial.distance import directed_hausdorff

# 1. Device and Hardware Inspection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 80)
print(" SYNTHRAD2023 MODEL 4: CROSS-FITTED RESIDUAL CONDITIONAL DIFFUSION ")
print("=" * 80)
print(f"PyTorch Version  : {torch.__version__}")
print(f"Compute Device   : {device}")
if torch.cuda.is_available():
    print(f"GPU Model        : {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Total VRAM       : {vram_gb:.2f} GB")
    torch.backends.cudnn.benchmark = True
else:
    print("WARNING: CUDA not available. Running on CPU (training will be slow).")

# 2. Strict Deterministic Random Seeding (Seed 42)
SEED = 42
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
print(f"[OK] Deterministic random seed locked to SEED = {SEED}.")

# 3. Output Directory Hierarchy
BASE_OUT_DIR = "/kaggle/working/synthrad_model4_residual_diffusion" if os.path.exists("/kaggle") else "./synthrad_model4_residual_diffusion"
DIR_CONFIGS     = os.path.join(BASE_OUT_DIR, "configs")
DIR_CHECKPOINTS = os.path.join(BASE_OUT_DIR, "checkpoints")
DIR_CKPT_COARSE = os.path.join(DIR_CHECKPOINTS, "coarse_resunet")
DIR_CKPT_DIFF   = os.path.join(DIR_CHECKPOINTS, "residual_diffusion")
DIR_METRICS     = os.path.join(BASE_OUT_DIR, "metrics")
DIR_FIGURES     = os.path.join(BASE_OUT_DIR, "figures")
DIR_PREDICTIONS = os.path.join(BASE_OUT_DIR, "predictions")
DIR_LOGS        = os.path.join(BASE_OUT_DIR, "logs")
DIR_REPORTS     = os.path.join(BASE_OUT_DIR, "reports")

for d in [DIR_CONFIGS, DIR_CHECKPOINTS, DIR_CKPT_COARSE, DIR_CKPT_DIFF, DIR_METRICS, DIR_FIGURES, DIR_PREDICTIONS, DIR_LOGS, DIR_REPORTS]:
    os.makedirs(d, exist_ok=True)

print(f"[OK] Isolated output hierarchy created at: {BASE_OUT_DIR}")
print("=" * 80)


In [ ]:
# =====================================================================
# CELL 2: EXPERIMENT CONFIGURATION & MODULAR EXECUTION FLAGS
# =====================================================================

# Modular Execution Flags (Allows resuming completed stages without retraining)
RUN_COARSE_TRAINING       = True   # Set to True to train 4-fold coarse ResUNet (skips if checkpoints exist)
RUN_RESIDUAL_TRAINING     = True   # Set to True to train residual conditional diffusion
RUN_OPTIONAL_ABLATIONS    = False  # Set to True to run ablations E0, E2, E4, E5
RUN_FINAL_TEST_EVALUATION = False  # Strictly False by default; only enable after final model freezing

CONFIG = {
    "project_name": "SynthRAD2023 Brain MRI-to-CT Synthesis",
    "experiment_name": "Model 4: Cross-Fitted ResUNet Residual Conditional Diffusion",
    "cohort": "Task 1 Brain (UMC Utrecht cohort, 60 patients)",
    "base_output_dir": BASE_OUT_DIR,
    "execution_flags": {
        "run_coarse_training": RUN_COARSE_TRAINING,
        "run_residual_training": RUN_RESIDUAL_TRAINING,
        "run_optional_ablations": RUN_OPTIONAL_ABLATIONS,
        "run_final_test_evaluation": RUN_FINAL_TEST_EVALUATION
    },
    "dataset": {
        "candidate_paths": [
            "/kaggle/input/datasets/lohithkgowda/synthrad2023-brain-2d-png/synthrad_png",
            "/kaggle/input/synthrad2023-brain-2d-png/synthrad_png",
            "./synthrad_png",
            "../synthrad_png",
            "synthrad_experiments/baselines/resunet/data"
        ],
        "image_size": 128,
        "intensity_range": [-1.0, 1.0],
        "exterior_background_fill": -1.0,
        "mask_source": "Strictly MRI-derived (threshold > 15, 7x7 ellipse closing, filled largest contour)",
        "num_workers": 0
    },
    "patient_splits": {
        "seed": 42,
        "isolation_policy": "Strict Patient-Level Isolation (Zero Slice Leakage)",
        "train_patients_count": 48,
        "val_patients_count": 6,
        "test_patients_count": 6,
        "train_expected_slices": 2879,
        "val_expected_slices": 372,
        "test_expected_slices": 363,
        "total_expected_slices": 3614,
        "n_folds": 4,
        "patients_per_fold": 12
    },
    "stage1_coarse_resunet": {
        "in_channels": 1,
        "out_channels": 1,
        "base_filters": 32,
        "epochs_per_fold": 40,
        "batch_size": 8,
        "learning_rate": 0.0001,
        "weight_decay": 0.0001,
        "loss": "Masked L1 + 0.10 * Masked SSIM",
        "output_activation": "Tanh [-1.0, 1.0]"
    },
    "stage2_residual_diffusion": {
        "in_channels": 4,  # [z_t, MRI, Coarse_CT, Mask]
        "out_channels": 1, # Predicted Gaussian noise epsilon
        "base_channels": 32,
        "time_emb_dim": 64,
        "epochs": 80,
        "batch_size": 8,
        "learning_rate": 0.0001,
        "weight_decay": 0.0001,
        "optimizer": "AdamW",
        "lr_scheduler": "CosineAnnealingLR (eta_min=1e-6)",
        "grad_clip_max_norm": 1.0,
        "loss_function": "MSE(predicted_noise, true_noise)",
        "mixed_precision": True,
        "checkpoint_interval_epochs": 5,
        "validation_interval_epochs": 5,
        "timesteps": 1000,
        "beta_start": 0.0001,
        "beta_end": 0.02,
        "beta_schedule": "linear",
        "ddim_inference_steps": 50,
        "ddim_eta": 0.0
    },
    "provisional_hu_mapping": {
        "status": "UNVERIFIED_PROVISIONAL",
        "disclaimer": "PNG intensities [0, 255] are normalized to [-1, 1]. Physical HU mapping is uncalibrated and reported provisionally for comparison purposes only.",
        "assumed_hu_min": -1000.0,
        "assumed_hu_max": 1500.0,
        "assumed_hu_range": 2500.0,
        "bone_threshold_hu": 300.0
    }
}

config_save_path = os.path.join(DIR_CONFIGS, "model4_experiment_config.json")
with open(config_save_path, "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)

print(f"[OK] Model 4 Experiment Configuration serialized to: {config_save_path}")


In [ ]:
# =====================================================================
# CELL 3: DATASET DISCOVERY & EXACT PATIENT SPLIT ASSERTIONS
# =====================================================================

def locate_dataset_root(candidate_paths):
    for path in candidate_paths:
        if os.path.exists(path) and os.path.isdir(os.path.join(path, "ct")) and os.path.isdir(os.path.join(path, "mri")):
            return os.path.abspath(path)
    for match in glob.glob("/kaggle/input/**/synthrad_png", recursive=True):
        if os.path.isdir(os.path.join(match, "ct")) and os.path.isdir(os.path.join(match, "mri")):
            return os.path.abspath(match)
    return None

DATASET_ROOT = locate_dataset_root(CONFIG["dataset"]["candidate_paths"])
if DATASET_ROOT is None:
    raise FileNotFoundError(
        "Could not locate SynthRAD2023 dataset directory. "
        "Please attach the Kaggle dataset 'lohithkgowda/synthrad2023-brain-2d-png' "
        "or ensure 'synthrad_png' with 'mri' and 'ct' subdirectories is present."
    )

print(f"[OK] SynthRAD2023 Dataset Root: {DATASET_ROOT}")
ct_dir  = os.path.join(DATASET_ROOT, "ct")
mri_dir = os.path.join(DATASET_ROOT, "mri")

all_ct_files  = sorted([f for f in os.listdir(ct_dir) if f.endswith(".png")])
all_mri_files = sorted([f for f in os.listdir(mri_dir) if f.endswith(".png")])

common_files = sorted(list(set(all_ct_files).intersection(set(all_mri_files))))
assert len(common_files) == len(all_ct_files) == len(all_mri_files), (
    f"Mismatch in filenames: CT ({len(all_ct_files)}), MRI ({len(all_mri_files)}), Common ({len(common_files)})"
)
print(f"[OK] Total paired slices matched: {len(common_files):,}")

# Group slices by patient ID (prefix before first underscore)
patient_to_slices = {}
for fname in common_files:
    pid = fname.split("_")[0]
    patient_to_slices.setdefault(pid, []).append(fname)

total_patients = len(patient_to_slices)
print(f"[OK] Total unique patients identified: {total_patients}")

# Exact Fixed Patient Split Lists (Do not modify or re-randomize)
EXPECTED_TRAIN_PIDS = sorted([
    "1BA001", "1BA005", "1BA012", "1BA014", "1BA022", "1BA032", "1BA040", "1BA058",
    "1BA075", "1BA082", "1BA085", "1BA091", "1BA100", "1BA103", "1BA105", "1BA116",
    "1BA131", "1BA143", "1BA151", "1BA158", "1BA159", "1BA164", "1BA175", "1BA184",
    "1BA185", "1BA189", "1BA201", "1BA206", "1BA220", "1BA222", "1BA227", "1BA239",
    "1BA247", "1BA253", "1BA260", "1BA266", "1BA278", "1BA288", "1BA292", "1BA294",
    "1BA305", "1BA325", "1BA328", "1BA336", "1BA345", "1BA358", "1BA359", "1BA368"
])
EXPECTED_VAL_PIDS = sorted(["1BA076", "1BA111", "1BA125", "1BA141", "1BA300", "1BA379"])
EXPECTED_TEST_PIDS = sorted(["1BA054", "1BA097", "1BA172", "1BA234", "1BA256", "1BA307"])

# Assert exact patient lists match
discovered_pids = set(patient_to_slices.keys())
assert set(EXPECTED_TRAIN_PIDS).issubset(discovered_pids), "Missing expected training patients!"
assert set(EXPECTED_VAL_PIDS).issubset(discovered_pids), "Missing expected validation patients!"
assert set(EXPECTED_TEST_PIDS).issubset(discovered_pids), "Missing expected test patients!"

# Assert mutual exclusivity
assert set(EXPECTED_TRAIN_PIDS).isdisjoint(set(EXPECTED_VAL_PIDS)), "Train and Val patient sets overlap!"
assert set(EXPECTED_TRAIN_PIDS).isdisjoint(set(EXPECTED_TEST_PIDS)), "Train and Test patient sets overlap!"
assert set(EXPECTED_VAL_PIDS).isdisjoint(set(EXPECTED_TEST_PIDS)), "Val and Test patient sets overlap!"

def gather_paths(pids):
    mri_paths, ct_paths = [], []
    for pid in pids:
        for sname in sorted(patient_to_slices[pid]):
            mri_paths.append(os.path.join(mri_dir, sname))
            ct_paths.append(os.path.join(ct_dir, sname))
    return mri_paths, ct_paths

train_mri, train_ct = gather_paths(EXPECTED_TRAIN_PIDS)
val_mri, val_ct     = gather_paths(EXPECTED_VAL_PIDS)
test_mri, test_ct   = gather_paths(EXPECTED_TEST_PIDS)

print("=" * 80)
print("VERIFYING EXACT FIXED PATIENT SPLIT SLICE COUNTS:")
print(f"  - Training Set   : {len(EXPECTED_TRAIN_PIDS)} patients, {len(train_mri):,} slices (Expected: 2,879)")
print(f"  - Validation Set : {len(EXPECTED_VAL_PIDS)} patients, {len(val_mri):,} slices (Expected: 372)")
print(f"  - Test Set       : {len(EXPECTED_TEST_PIDS)} patients, {len(test_mri):,} slices (Expected: 363)")
print("=" * 80)

assert len(train_mri) == CONFIG["patient_splits"]["train_expected_slices"], f"Train slice count mismatch: {len(train_mri)}"
assert len(val_mri)   == CONFIG["patient_splits"]["val_expected_slices"],   f"Val slice count mismatch: {len(val_mri)}"
assert len(test_mri)  == CONFIG["patient_splits"]["test_expected_slices"],  f"Test slice count mismatch: {len(test_mri)}"

print("[PASS] All patient partitions and exact slice counts validated successfully.")


In [ ]:
# =====================================================================
# CELL 4: SHARED PREPROCESSING & DATASET DEFINITIONS
# =====================================================================

def derive_mri_body_mask(img_arr, threshold=15.0):
    """
    Derives anatomical head contour mask STRICTLY from MRI.
    - Zero ground-truth CT information is used.
    - Threshold grayscale MRI > 15.0.
    - Morphological closing with a 7x7 elliptical structuring element.
    - Retains and fills the largest external contour.
    """
    binary = (img_arr > threshold).astype(np.uint8) * 255
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask = np.zeros_like(img_arr, dtype=np.uint8)
    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        cv2.drawContours(mask, [largest_contour], -1, 255, thickness=cv2.FILLED)
    else:
        mask = closed
    return (mask > 0).astype(np.float32)

class SynthRADPairedDataset(Dataset):
    """
    Standard paired dataset loading MRI, CT, and MRI-derived mask.
    Supports synchronous geometric augmentation for training.
    """
    def __init__(self, mri_paths, ct_paths, image_size=128, is_train=False):
        assert len(mri_paths) == len(ct_paths), "Mismatch between MRI and CT paths count"
        self.mri_paths  = mri_paths
        self.ct_paths   = ct_paths
        self.image_size = image_size
        self.is_train   = is_train

    def __len__(self):
        return len(self.mri_paths)

    def _process_slice(self, mri_path, ct_path):
        mri_img = Image.open(mri_path).convert('L')
        ct_img  = Image.open(ct_path).convert('L')
        
        mri_arr = np.array(mri_img, dtype=np.float32)
        ct_arr  = np.array(ct_img,  dtype=np.float32)
        
        # Mask strictly from MRI
        mri_mask = derive_mri_body_mask(mri_arr)
        mri_arr  = mri_arr * mri_mask
        
        # Resize: bilinear for images, nearest for mask
        mri_res  = cv2.resize(mri_arr,  (self.image_size, self.image_size), interpolation=cv2.INTER_LINEAR)
        ct_res   = cv2.resize(ct_arr,   (self.image_size, self.image_size), interpolation=cv2.INTER_LINEAR)
        mask_res = cv2.resize(mri_mask, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)
        
        # Normalize: [0, 255] -> [-1.0, 1.0]
        mri_norm = (mri_res / 255.0) * 2.0 - 1.0
        ct_norm  = (ct_res  / 255.0) * 2.0 - 1.0
        
        # Air exterior set to -1.0
        mri_norm[mask_res < 0.5] = -1.0
        ct_norm[mask_res < 0.5]  = -1.0
        
        t_mri  = torch.from_numpy(mri_norm).unsqueeze(0).float()
        t_ct   = torch.from_numpy(ct_norm).unsqueeze(0).float()
        t_mask = torch.from_numpy(mask_res).unsqueeze(0).float()
        
        return t_mri, t_ct, t_mask

    def __getitem__(self, idx):
        mri, ct, mask = self._process_slice(self.mri_paths[idx], self.ct_paths[idx])
        
        if self.is_train:
            if random.random() > 0.5:
                mri  = TF.hflip(mri)
                ct   = TF.hflip(ct)
                mask = TF.hflip(mask)
            angle = random.uniform(-6.0, 6.0)
            tx = int(random.uniform(-3, 3))
            ty = int(random.uniform(-3, 3))
            scale = random.uniform(0.97, 1.03)
            mri  = TF.affine(mri,  angle=angle, translate=[tx, ty], scale=scale, shear=0, fill=-1.0)
            ct   = TF.affine(ct,   angle=angle, translate=[tx, ty], scale=scale, shear=0, fill=-1.0)
            mask = TF.affine(mask, angle=angle, translate=[tx, ty], scale=scale, shear=0, fill=0.0)

        pid   = os.path.basename(self.mri_paths[idx]).split("_")[0]
        sname = os.path.basename(self.mri_paths[idx])

        return {
            "mri": mri,
            "ct":  ct,
            "mask": mask,
            "patient_id": pid,
            "slice_name": sname
        }

BATCH_SIZE  = CONFIG["stage1_coarse_resunet"]["batch_size"]
IMG_SIZE    = CONFIG["dataset"]["image_size"]
NUM_WORKERS = CONFIG["dataset"]["num_workers"]

val_dataset_eval  = SynthRADPairedDataset(val_mri,  val_ct,  image_size=IMG_SIZE, is_train=False)
test_dataset_eval = SynthRADPairedDataset(test_mri, test_ct, image_size=IMG_SIZE, is_train=False)
val_loader        = DataLoader(val_dataset_eval, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"[OK] Validation Loader instantiated: {len(val_loader)} batches ({len(val_dataset_eval)} slices).")


In [ ]:
# =====================================================================
# CELL 5: STAGE 1 — RESUNET ARCHITECTURE & SUPERVISED RECONSTRUCTION LOSS
# =====================================================================

class ResBlock(nn.Module):
    """Residual convolutional block with GroupNorm, LeakyReLU, and 1x1 shortcut."""
    def __init__(self, in_channels, out_channels, num_groups=8):
        super().__init__()
        groups = num_groups if out_channels % num_groups == 0 else (4 if out_channels % 4 == 0 else 1)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.norm1 = nn.GroupNorm(groups, out_channels)
        self.act1  = nn.LeakyReLU(0.2, inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.norm2 = nn.GroupNorm(groups, out_channels)
        self.act2  = nn.LeakyReLU(0.2, inplace=True)
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.GroupNorm(groups, out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        residual = self.shortcut(x)
        h = self.act1(self.norm1(self.conv1(x)))
        h = self.norm2(self.conv2(h))
        return self.act2(h + residual)

class ResUNet(nn.Module):
    """
    Stage 1 Coarse ResUNet baseline architecture (Model 1 reference).
    Outputs coarse synthetic CT in [-1.0, 1.0] via Tanh activation.
    """
    def __init__(self, in_channels=1, out_channels=1, base_filters=32):
        super().__init__()
        f = base_filters
        self.enc1 = ResBlock(in_channels, f)        # 128 -> 128
        self.pool1 = nn.MaxPool2d(2)                # 128 -> 64
        self.enc2 = ResBlock(f, f * 2)              # 64 -> 64
        self.pool2 = nn.MaxPool2d(2)                # 64 -> 32
        self.enc3 = ResBlock(f * 2, f * 4)          # 32 -> 32
        self.pool3 = nn.MaxPool2d(2)                # 32 -> 16
        self.enc4 = ResBlock(f * 4, f * 8)          # 16 -> 16
        self.pool4 = nn.MaxPool2d(2)                # 16 -> 8

        self.bottleneck = nn.Sequential(
            ResBlock(f * 8, f * 8),
            ResBlock(f * 8, f * 8)
        )

        self.up4 = nn.ConvTranspose2d(f * 8, f * 4, kernel_size=2, stride=2)
        self.dec4 = ResBlock(f * 8 + f * 4, f * 4)
        self.up3 = nn.ConvTranspose2d(f * 4, f * 2, kernel_size=2, stride=2)
        self.dec3 = ResBlock(f * 4 + f * 2, f * 2)
        self.up2 = nn.ConvTranspose2d(f * 2, f, kernel_size=2, stride=2)
        self.dec2 = ResBlock(f * 2 + f, f)
        self.up1 = nn.ConvTranspose2d(f, f, kernel_size=2, stride=2)
        self.dec1 = ResBlock(f + f, f)

        self.final_conv = nn.Conv2d(f, out_channels, kernel_size=1)
        self.final_act = nn.Tanh()

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))
        b = self.bottleneck(self.pool4(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.final_act(self.final_conv(d1))

class DifferentiableSSIM(nn.Module):
    """Differentiable SSIM loss for normalized [-1.0, 1.0] domain."""
    def __init__(self, window_size=11, sigma=1.5, data_range=2.0):
        super().__init__()
        self.window_size = window_size
        self.data_range = data_range
        coords = torch.arange(window_size, dtype=torch.float32) - window_size // 2
        g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
        g = g / g.sum()
        _2d = torch.outer(g, g).unsqueeze(0).unsqueeze(0)
        self.register_buffer("window", _2d)

    def forward(self, img1, img2, mask=None):
        w = self.window.to(dtype=img1.dtype, device=img1.device)
        pad = self.window_size // 2
        mu1 = F.conv2d(img1, w, padding=pad)
        mu2 = F.conv2d(img2, w, padding=pad)
        mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1 * mu2
        sigma1_sq = F.conv2d(img1 * img1, w, padding=pad) - mu1_sq
        sigma2_sq = F.conv2d(img2 * img2, w, padding=pad) - mu2_sq
        sigma12   = F.conv2d(img1 * img2, w, padding=pad) - mu1_mu2
        c1 = (0.01 * self.data_range) ** 2
        c2 = (0.03 * self.data_range) ** 2
        ssim_map = ((2 * mu1_mu2 + c1) * (2 * sigma12 + c2)) / (
            (mu1_sq + mu2_sq + c1) * (sigma1_sq + sigma2_sq + c2) + 1e-8
        )
        if mask is not None:
            m = mask.to(dtype=img1.dtype, device=img1.device)
            return ((1.0 - ssim_map) * m).sum() / (m.sum() + 1e-8)
        return 1.0 - ssim_map.mean()

class ResUNetSupervisedLoss(nn.Module):
    """L_total = Masked_L1 + lambda_ssim * Masked_SSIM."""
    def __init__(self, lambda_ssim=0.10):
        super().__init__()
        self.lambda_ssim = lambda_ssim
        self.ssim_loss = DifferentiableSSIM(window_size=11, data_range=2.0)

    def forward(self, pred, target, mask):
        diff = torch.abs(pred - target)
        m_sum = mask.sum() + 1e-8
        l1 = (diff * mask).sum() / m_sum
        l_ssim = self.ssim_loss(pred, target, mask=mask)
        return l1 + self.lambda_ssim * l_ssim

coarse_test = ResUNet(base_filters=CONFIG["stage1_coarse_resunet"]["base_filters"])
coarse_params = sum(p.numel() for p in coarse_test.parameters() if p.requires_grad)
print(f"[OK] Stage 1 ResUNet architecture verified: {coarse_params:,} parameters (~{coarse_params/1e6:.2f}M).")


In [ ]:
# =====================================================================
# CELL 6: PATIENT-LEVEL 4-FOLD CROSS-FITTING PARTITION
# =====================================================================

# 48 training patients partitioned deterministically into 4 disjoint folds of 12 patients each
N_FOLDS = CONFIG["patient_splits"]["n_folds"]
PATIENTS_PER_FOLD = CONFIG["patient_splits"]["patients_per_fold"]

# Explicit 4-fold partition (Seed 42 deterministic chunking)
FOLDS = {}
for k in range(N_FOLDS):
    fold_pids = EXPECTED_TRAIN_PIDS[k * PATIENTS_PER_FOLD : (k + 1) * PATIENTS_PER_FOLD]
    FOLDS[k + 1] = fold_pids

# Verify disjoint partition covering all 48 patients
all_fold_pids = []
for k, pids in FOLDS.items():
    all_fold_pids.extend(pids)

assert sorted(all_fold_pids) == EXPECTED_TRAIN_PIDS, "Fold partition does not match exact training patient list!"
for k1 in range(1, N_FOLDS + 1):
    for k2 in range(k1 + 1, N_FOLDS + 1):
        assert set(FOLDS[k1]).isdisjoint(set(FOLDS[k2])), f"Folds {k1} and {k2} share patients!"

print("=" * 80)
print("PATIENT-LEVEL 4-FOLD CROSS-FITTING PARTITIONS:")
print("=" * 80)
fold_slice_counts = {}
for k, pids in FOLDS.items():
    m_paths, c_paths = gather_paths(pids)
    fold_slice_counts[k] = len(m_paths)
    print(f"  Fold {k}: {len(pids)} patients ({pids[0]} ... {pids[-1]}), {len(m_paths):,} slices")

total_fold_slices = sum(fold_slice_counts.values())
assert total_fold_slices == CONFIG["patient_splits"]["train_expected_slices"], "Total fold slices mismatch!"
print(f"Total slices across all 4 folds: {total_fold_slices:,} (Matches expected 2,879)")
print("[PASS] 4-Fold cross-fitting partition validated with strict patient isolation.")

# Save fold configuration
fold_config = {"fold_patients": FOLDS, "fold_slice_counts": fold_slice_counts}
with open(os.path.join(DIR_CONFIGS, "cross_fitting_folds.json"), "w", encoding="utf-8") as f:
    json.dump(fold_config, f, indent=2)


In [ ]:
# =====================================================================
# CELL 7: STAGE 1 — 4-FOLD COARSE RESUNET TRAINING ENGINE
# =====================================================================

COARSE_EPOCHS = CONFIG["stage1_coarse_resunet"]["epochs_per_fold"]
COARSE_LR     = CONFIG["stage1_coarse_resunet"]["learning_rate"]
COARSE_WD     = CONFIG["stage1_coarse_resunet"]["weight_decay"]

def train_coarse_fold(fold_idx, train_pids_fold, val_pids_fold):
    """Trains a ResUNet on 3 folds (36 patients) and saves the best checkpoint on held-out fold."""
    ckpt_path = os.path.join(DIR_CKPT_COARSE, f"coarse_resunet_fold_{fold_idx}.pth")
    
    # Resumption check: If already trained, skip retraining
    if os.path.exists(ckpt_path):
        print(f"[FOUND] Existing checkpoint for Fold {fold_idx} at: {ckpt_path}. Skipping retraining.")
        model = ResUNet(base_filters=CONFIG["stage1_coarse_resunet"]["base_filters"]).to(device)
        data = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(data["model_state_dict"])
        return model, data.get("best_val_score", 0.0)

    print("")
    print(f"--> Training Coarse ResUNet Fold {fold_idx}/{N_FOLDS} ({len(train_pids_fold)} Train PIDs, {len(val_pids_fold)} Val PIDs)...")
    tr_m, tr_c = gather_paths(train_pids_fold)
    va_m, va_c = gather_paths(val_pids_fold)
    
    ds_tr = SynthRADPairedDataset(tr_m, tr_c, image_size=IMG_SIZE, is_train=True)
    ds_va = SynthRADPairedDataset(va_m, va_c, image_size=IMG_SIZE, is_train=False)
    
    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
    dl_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    model = ResUNet(base_filters=CONFIG["stage1_coarse_resunet"]["base_filters"]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=COARSE_LR, weight_decay=COARSE_WD)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=COARSE_EPOCHS, eta_min=1e-6)
    criterion = ResUNetSupervisedLoss(lambda_ssim=0.10)
    scaler = torch.amp.GradScaler('cuda', enabled=(device.type == "cuda"))

    best_val_loss = float("inf")
    best_epoch = 0

    for epoch in range(1, COARSE_EPOCHS + 1):
        model.train()
        running_loss = 0.0
        for batch in dl_tr:
            mri  = batch["mri"].to(device)
            ct   = batch["ct"].to(device)
            mask = batch["mask"].to(device)
            
            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
                pred = model(mri)
                loss = criterion(pred, ct, mask)
                
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
            
        scheduler.step()
        ep_tr_loss = running_loss / len(dl_tr)

        # Validation on held-out fold
        model.eval()
        running_val_l1 = 0.0
        with torch.no_grad():
            for batch in dl_va:
                mri  = batch["mri"].to(device)
                ct   = batch["ct"].to(device)
                mask = batch["mask"].to(device)
                pred = model(mri)
                diff = torch.abs(pred - ct)
                running_val_l1 += ((diff * mask).sum() / (mask.sum() + 1e-8)).item()
                
        ep_val_l1 = running_val_l1 / len(dl_va)
        if ep_val_l1 < best_val_loss:
            best_val_loss = ep_val_l1
            best_epoch = epoch
            torch.save({
                "fold": fold_idx,
                "epoch": best_epoch,
                "best_val_score": best_val_loss,
                "model_state_dict": model.state_dict(),
                "config": CONFIG["stage1_coarse_resunet"]
            }, ckpt_path)

        if epoch % 10 == 0 or epoch == COARSE_EPOCHS:
            print(f"  Fold {fold_idx} | Epoch [{epoch:02d}/{COARSE_EPOCHS}] | Train Loss: {ep_tr_loss:.4f} | Val L1: {ep_val_l1:.4f} (Best: {best_val_loss:.4f} @ Ep {best_epoch})")

    # Load best checkpoint
    best_data = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(best_data["model_state_dict"])
    return model, best_val_loss

COARSE_MODELS = {}
if RUN_COARSE_TRAINING:
    print("=" * 80)
    print("STARTING 4-FOLD CROSS-FITTED RESUNET COARSE MODEL TRAINING")
    print("=" * 80)
    for k in range(1, N_FOLDS + 1):
        val_pids_k = FOLDS[k]
        train_pids_k = [pid for j, pids in FOLDS.items() if j != k for pid in pids]
        m_k, score_k = train_coarse_fold(k, train_pids_k, val_pids_k)
        COARSE_MODELS[k] = m_k
    print("[PASS] All 4 coarse fold models trained and verified.")
else:
    print("[INFO] RUN_COARSE_TRAINING is set to False. Attempting to load existing fold checkpoints...")
    for k in range(1, N_FOLDS + 1):
        ckpt_path = os.path.join(DIR_CKPT_COARSE, f"coarse_resunet_fold_{k}.pth")
        if os.path.exists(ckpt_path):
            m_k = ResUNet(base_filters=CONFIG["stage1_coarse_resunet"]["base_filters"]).to(device)
            m_k.load_state_dict(torch.load(ckpt_path, map_location=device)["model_state_dict"])
            m_k.eval()
            COARSE_MODELS[k] = m_k
            print(f"  [LOADED] Fold {k} from {ckpt_path}")
        else:
            print(f"  [MISSING] Checkpoint for Fold {k} not found. Set RUN_COARSE_TRAINING = True.")


In [ ]:
# =====================================================================
# CELL 8: OUT-OF-FOLD (OOF) COARSE CT PREDICTION GENERATION & CACHING
# =====================================================================

OOF_CACHE_FILE = os.path.join(DIR_PREDICTIONS, "oof_coarse_predictions_dict.pt")

if os.path.exists(OOF_CACHE_FILE):
    print(f"[FOUND] Loading precomputed OOF coarse predictions from: {OOF_CACHE_FILE}")
    OOF_PREDICTIONS = torch.load(OOF_CACHE_FILE, map_location="cpu")
    print(f"[OK] Loaded OOF coarse predictions for {len(OOF_PREDICTIONS):,} training slices.")
else:
    print("=" * 80)
    print("GENERATING LEAKAGE-FREE OUT-OF-FOLD (OOF) COARSE PREDICTIONS")
    print("=" * 80)
    assert len(COARSE_MODELS) == N_FOLDS, f"Expected {N_FOLDS} trained fold models, found {len(COARSE_MODELS)}"
    
    OOF_PREDICTIONS = {}
    
    # For each fold k, generate coarse predictions for its held-out patients using model k
    for k in range(1, N_FOLDS + 1):
        held_out_pids = FOLDS[k]
        m_paths, c_paths = gather_paths(held_out_pids)
        ds_k = SynthRADPairedDataset(m_paths, c_paths, image_size=IMG_SIZE, is_train=False)
        dl_k = DataLoader(ds_k, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
        
        model_k = COARSE_MODELS[k].to(device)
        model_k.eval()
        
        with torch.no_grad():
            for batch in tqdm(dl_k, desc=f"Predicting OOF Fold {k}/{N_FOLDS}"):
                mri   = batch["mri"].to(device)
                mask  = batch["mask"].to(device)
                snames = batch["slice_name"]
                
                # Predict coarse CT: [B, 1, H, W] in [-1, 1]
                coarse = model_k(mri)
                # Enforce air exterior strictly to -1.0
                coarse = torch.where(mask > 0.5, coarse, torch.full_like(coarse, -1.0))
                
                for i in range(mri.size(0)):
                    sname = snames[i]
                    OOF_PREDICTIONS[sname] = coarse[i].cpu()
                    
    # Assert exact completeness
    assert len(OOF_PREDICTIONS) == CONFIG["patient_splits"]["train_expected_slices"], (
        f"OOF slice count mismatch: {len(OOF_PREDICTIONS)} vs {CONFIG['patient_splits']['train_expected_slices']}"
    )
    
    # Cache to disk for instant resumption
    torch.save(OOF_PREDICTIONS, OOF_CACHE_FILE)
    print(f"[PASS] Successfully cached {len(OOF_PREDICTIONS):,} OOF coarse predictions to: {OOF_CACHE_FILE}")


In [ ]:
# =====================================================================
# CELL 9: RESIDUAL ANALYSIS & FIXED RESIDUAL SCALE ESTIMATION
# =====================================================================

print("=" * 80)
print("COMPUTING OUT-OF-FOLD RESIDUAL STATISTICS AND ESTIMATING SCALE s")
print("=" * 80)

# Build a temporary dataset to evaluate OOF residuals across all 48 training patients
train_paired_eval = SynthRADPairedDataset(train_mri, train_ct, image_size=IMG_SIZE, is_train=False)
eval_loader_train = DataLoader(train_paired_eval, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

residual_masked_values = []
coarse_mae_list = []

for batch in tqdm(eval_loader_train, desc="Analyzing OOF Residuals"):
    ct    = batch["ct"]
    mask  = batch["mask"]
    snames = batch["slice_name"]
    
    for i in range(ct.size(0)):
        sname = snames[i]
        c_pred = OOF_PREDICTIONS[sname] # [1, H, W]
        gt_ct  = ct[i]                 # [1, H, W]
        m_arr  = mask[i]               # [1, H, W]
        
        # Residual: r = CT_target - CT_coarse_OOF
        r = gt_ct - c_pred
        
        body = m_arr > 0.5
        if body.sum() > 0:
            residual_masked_values.extend(r[body].numpy().tolist())
            coarse_mae_list.append(torch.abs(r[body]).mean().item())

res_np = np.array(residual_masked_values, dtype=np.float32)

res_mean = float(np.mean(res_np))
res_std  = float(np.std(res_np))
res_p01  = float(np.percentile(res_np, 1.0))
res_p99  = float(np.percentile(res_np, 99.0))
coarse_oof_mae = float(np.mean(coarse_mae_list))

# Fixed residual scale s is the standard deviation of anatomical residuals
# Normalized residual: z_0 = r / s
RESIDUAL_SCALE = res_std

print(f"OOF Residual Distribution inside Anatomical Mask (N={len(res_np):,} pixels):")
print(f"  - Mean Residual       : {res_mean:.5f}")
print(f"  - Residual Std Dev (s): {res_std:.5f}  <-- Fixed Residual Scale")
print(f"  - 1st Percentile      : {res_p01:.5f}")
print(f"  - 99th Percentile     : {res_p99:.5f}")
print(f"  - Coarse OOF CT MAE   : {coarse_oof_mae:.5f} (Normalized Intensity)")
print("=" * 80)

# Save residual scale configuration
scale_config = {
    "residual_scale_s": RESIDUAL_SCALE,
    "residual_mean": res_mean,
    "residual_std": res_std,
    "percentile_01": res_p01,
    "percentile_99": res_p99,
    "coarse_oof_mae": coarse_oof_mae
}
with open(os.path.join(DIR_CONFIGS, "residual_scale_config.json"), "w", encoding="utf-8") as f:
    json.dump(scale_config, f, indent=2)

# Plot residual distribution
plt.figure(figsize=(7, 3.5), dpi=100)
plt.hist(res_np[::10], bins=100, color="teal", alpha=0.75, density=True)
plt.axvline(0.0, color="black", linestyle="--", lw=1)
plt.axvline(res_std, color="red", linestyle=":", label=f"+1 Std (s={res_std:.3f})")
plt.axvline(-res_std, color="red", linestyle=":", label=f"-1 Std (s={res_std:.3f})")
plt.title("Distribution of Out-of-Fold Residuals r = CT - Coarse_OOF")
plt.xlabel("Residual Value (r)")
plt.ylabel("Density")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(DIR_FIGURES, "oof_residual_distribution.png"))
plt.show()


In [ ]:
# =====================================================================
# CELL 10: STAGE 2 — RESIDUAL CONDITIONAL U-NET DENOISER ARCHITECTURE
# =====================================================================

class SinusoidalPositionEmbeddings(nn.Module):
    """Standard sinusoidal timestep embeddings."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000.0) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        return torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)

class DiffusionResBlock(nn.Module):
    """
    Standard Diffusion Residual Block with GroupNorm, SiLU, and time injection.
    Maintains identity residual connection: h = conv2(act(conv1(x) + time_emb)) + shortcut(x).
    """
    def __init__(self, in_ch, out_ch, time_emb_dim):
        super().__init__()
        self.time_mlp = nn.Linear(time_emb_dim, out_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
        self.gn1   = nn.GroupNorm(8 if out_ch % 8 == 0 else 4, out_ch)
        self.act1  = nn.SiLU()
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1)
        self.gn2   = nn.GroupNorm(8 if out_ch % 8 == 0 else 4, out_ch)
        self.act2  = nn.SiLU()
        self.shortcut = nn.Conv2d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.act1(self.gn1(self.conv1(x)))
        h = h + self.act1(self.time_mlp(t_emb))[:, :, None, None]
        h = self.act2(self.gn2(self.conv2(h)))
        return h + self.shortcut(x)

class ResidualConditionalUNet(nn.Module):
    """
    Conditional U-Net Denoiser for Residual Diffusion (Model 4).
    - Inputs: 4 channels [z_t, MRI, Coarse_CT, mask].
    - Output: 1 unconstrained channel predicting Gaussian noise epsilon.
    - Downsampling: Strided Convolutions (kernel=3, stride=2, padding=1).
    - Upsampling: ConvTranspose2d (kernel=2, stride=2).
    - Residual connections in every block.
    - Parameters: ~3.46M.
    """
    def __init__(self, in_channels=4, out_channels=1, base_channels=32, time_emb_dim=64):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.base_channels = base_channels
        self.time_emb_dim = time_emb_dim

        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.SiLU()
        )
        c = base_channels

        # Encoder stages
        self.enc1  = DiffusionResBlock(in_channels, c, time_emb_dim)            # 128 -> 128
        self.down1 = nn.Conv2d(c, c, kernel_size=3, stride=2, padding=1)        # 128 -> 64
        self.enc2  = DiffusionResBlock(c, c * 2, time_emb_dim)                  # 64  -> 64
        self.down2 = nn.Conv2d(c * 2, c * 2, kernel_size=3, stride=2, padding=1)# 64  -> 32
        self.enc3  = DiffusionResBlock(c * 2, c * 4, time_emb_dim)              # 32  -> 32
        self.down3 = nn.Conv2d(c * 4, c * 4, kernel_size=3, stride=2, padding=1)# 32  -> 16

        # Bottleneck stages
        self.bot1  = DiffusionResBlock(c * 4, c * 8, time_emb_dim)              # 16  -> 16
        self.bot2  = DiffusionResBlock(c * 8, c * 8, time_emb_dim)              # 16  -> 16

        # Decoder stages with skip connections
        self.up3   = nn.ConvTranspose2d(c * 8, c * 4, kernel_size=2, stride=2)  # 16  -> 32
        self.dec3  = DiffusionResBlock(c * 4 + c * 4, c * 4, time_emb_dim)      # cat with enc3
        self.up2   = nn.ConvTranspose2d(c * 4, c * 2, kernel_size=2, stride=2)  # 32  -> 64
        self.dec2  = DiffusionResBlock(c * 2 + c * 2, c * 2, time_emb_dim)      # cat with enc2
        self.up1   = nn.ConvTranspose2d(c * 2, c, kernel_size=2, stride=2)      # 64  -> 128
        self.dec1  = DiffusionResBlock(c + c, c, time_emb_dim)                  # cat with enc1

        self.final_conv = nn.Conv2d(c, out_channels, kernel_size=1)

    def forward(self, x, t):
        assert x.shape[1] == self.in_channels, f"Expected {self.in_channels} channels, got {x.shape[1]}"
        t_emb = self.time_mlp(t)
        
        e1 = self.enc1(x, t_emb)
        d1 = self.down1(e1)
        e2 = self.enc2(d1, t_emb)
        d2 = self.down2(e2)
        e3 = self.enc3(d2, t_emb)
        d3 = self.down3(e3)

        b = self.bot2(self.bot1(d3, t_emb), t_emb)

        u3 = self.up3(b)
        d3_out = self.dec3(torch.cat([u3, e3], dim=1), t_emb)
        u2 = self.up2(d3_out)
        d2_out = self.dec2(torch.cat([u2, e2], dim=1), t_emb)
        u1 = self.up1(d2_out)
        d1_out = self.dec1(torch.cat([u1, e1], dim=1), t_emb)

        return self.final_conv(d1_out)

residual_diff_model = ResidualConditionalUNet(
    in_channels=CONFIG["stage2_residual_diffusion"]["in_channels"],
    out_channels=CONFIG["stage2_residual_diffusion"]["out_channels"],
    base_channels=CONFIG["stage2_residual_diffusion"]["base_channels"],
    time_emb_dim=CONFIG["stage2_residual_diffusion"]["time_emb_dim"]
).to(device)

diff_params = sum(p.numel() for p in residual_diff_model.parameters() if p.requires_grad)
print("=" * 80)
print(f"RESIDUAL CONDITIONAL U-NET DENOISER INITIALIZED")
print(f"Trainable Parameters : {diff_params:,} (~{diff_params/1e6:.2f}M)")
print(f"Input Channels       : 4 [z_t, MRI, Coarse_CT, Mask]")
print(f"Output Channels      : 1 [Predicted Noise epsilon]")
print("=" * 80)


In [ ]:
# =====================================================================
# CELL 11: RESIDUAL DIFFUSION SCHEDULER & MATHEMATICAL PREFLIGHT
# =====================================================================

class ResidualDDPMScheduler:
    """
    Standard DDPM noise scheduler for normalized residuals z_0 = r / s.
    Forward process: z_t = sqrt(alpha_bar_t) * z_0 + sqrt(1 - alpha_bar_t) * epsilon
    """
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02, device="cuda"):
        self.timesteps = timesteps
        self.device = torch.device(device) if isinstance(device, str) else device
        self.betas = torch.linspace(beta_start, beta_end, timesteps, dtype=torch.float32).to(self.device)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0).to(self.device)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)

    def add_noise(self, z_0, t):
        noise = torch.randn_like(z_0)
        alphas_cp = self.alphas_cumprod.to(z_0.device)
        sqrt_alpha = alphas_cp[t].sqrt().view(-1, 1, 1, 1)
        sqrt_one_minus = (1.0 - alphas_cp[t]).sqrt().view(-1, 1, 1, 1)
        return sqrt_alpha * z_0 + sqrt_one_minus * noise, noise

    def reconstruct_z0(self, z_t, noise, t):
        alphas_cp = self.alphas_cumprod.to(z_t.device)
        sqrt_alpha = alphas_cp[t].sqrt().view(-1, 1, 1, 1)
        sqrt_one_minus = (1.0 - alphas_cp[t]).sqrt().view(-1, 1, 1, 1)
        return (z_t - sqrt_one_minus * noise) / (sqrt_alpha + 1e-8)

sched = ResidualDDPMScheduler(
    timesteps=CONFIG["stage2_residual_diffusion"]["timesteps"],
    beta_start=CONFIG["stage2_residual_diffusion"]["beta_start"],
    beta_end=CONFIG["stage2_residual_diffusion"]["beta_end"],
    device=device
)

# 1. Schedule buffer checks (with numerical tolerance for float roundoff)
assert len(sched.betas) == 1000, "Betas length mismatch"
assert abs(sched.betas[0].item() - 1e-4) < 1e-6, "Beta start mismatch"
assert abs(sched.betas[-1].item() - 0.02) < 1e-5, "Beta end mismatch"
assert (sched.alphas_cumprod[:-1] >= sched.alphas_cumprod[1:]).all(), "Alpha-cumprod not decreasing"

# 2. Algebraic Inversion with True Noise across timesteps
dummy_z0 = torch.randn(2, 1, 128, 128, device=device)
timesteps_test = [0, 50, 250, 500, 750, 999]
inversion_errors = []
for t_val in timesteps_test:
    t_tens = torch.tensor([t_val, t_val], device=device, dtype=torch.long)
    z_t, true_noise = sched.add_noise(dummy_z0, t_tens)
    recon = sched.reconstruct_z0(z_t, true_noise, t_tens)
    err = (recon - dummy_z0).abs().max().item()
    inversion_errors.append(err)

assert max(inversion_errors) < 1e-4, f"Algebraic inversion failed: {max(inversion_errors)}"
print(f"[PASS] Residual diffusion schedule and true-noise algebraic inversion verified (< 1e-4 error across all timesteps).")


In [ ]:
# =====================================================================
# CELL 12: DETERMINISTIC RESIDUAL DDIM SAMPLER
# =====================================================================

@torch.no_grad()
def sample_residual_ddim(model, mri, coarse_ct, mask, residual_scale, ddim_steps=50, initial_noise=None, instrument=False):
    """
    Deterministic DDIM Reverse Sampler for Residual Diffusion (eta = 0.0).
    - Condition inputs: [z_t, MRI, Coarse_CT, mask] (4 channels).
    - Predicts noise epsilon pointing towards clean residual z_0 = r / s.
    - Note: z_0 is NOT clipped to [-1, 1] because residuals naturally exceed [-1, 1].
    - Final CT: CT_sct = clip(coarse_ct + s * z_0, -1, 1).
    - Exterior scanner air outside mask is restored strictly to -1.0.
    """
    model.eval()
    dev = mri.device
    b   = mri.shape[0]

    if initial_noise is not None:
        z_t = initial_noise.to(dev).clone()
    else:
        z_t = torch.randn_like(mri).to(dev)

    timesteps = sorted(torch.linspace(0, sched.timesteps - 1, ddim_steps).long().tolist(), reverse=True)
    alphas_cp = sched.alphas_cumprod.to(dev)
    diagnostics = []

    for i, t_val in enumerate(timesteps):
        t = torch.full((b,), t_val, device=dev, dtype=torch.long)
        
        # Denoiser conditioning: [z_t, MRI, Coarse_CT, mask]
        cond_in = torch.cat([z_t, mri, coarse_ct, mask], dim=1)
        pred_noise = model(cond_in, t)

        alpha_cur  = alphas_cp[t_val]
        alpha_prev = alphas_cp[timesteps[i + 1]] if i < len(timesteps) - 1 else torch.tensor(1.0, device=dev)

        # Raw predicted clean residual z_0 (No clean-CT [-1, 1] clamping applied to residuals)
        pred_z0 = (z_t - (1.0 - alpha_cur).sqrt() * pred_noise) / (alpha_cur.sqrt() + 1e-8)
        
        # Safeguard solely against extreme floating-point blowup (+/- 6 standard deviations)
        pred_z0 = torch.clamp(pred_z0, -6.0, 6.0)

        # Re-project direction vector to maintain collinearity along reverse trajectory
        pred_noise_adj = (z_t - alpha_cur.sqrt() * pred_z0) / ((1.0 - alpha_cur).sqrt() + 1e-8)
        dir_zt = (1.0 - alpha_prev).clamp(min=0.0).sqrt() * pred_noise_adj
        z_t = alpha_prev.sqrt() * pred_z0 + dir_zt

        if instrument and (i % max(1, ddim_steps // 5) == 0 or i == len(timesteps) - 1):
            diagnostics.append({
                "step": i,
                "timestep": t_val,
                "latent_z_mean": z_t.mean().item(),
                "latent_z_std": z_t.std().item(),
                "pred_noise_mean": pred_noise.mean().item(),
                "pred_noise_std": pred_noise.std().item()
            })

    # Final reconstruction: sCT = clip(Coarse + s * z_0, -1.0, 1.0)
    pred_residual = z_t * residual_scale
    sct = torch.clamp(coarse_ct + pred_residual, -1.0, 1.0)
    
    # Strictly enforce exterior background air to -1.0
    sct = torch.where(mask > 0.5, sct, torch.full_like(sct, -1.0))

    return (sct, pred_residual, diagnostics) if instrument else (sct, pred_residual)

print("[OK] Deterministic Residual DDIM Sampler initialized.")


In [ ]:
# =====================================================================
# CELL 13: RESIDUAL DIFFUSION DATASET WITH CACHED OOF RESIDUALS
# =====================================================================

class SynthRADResidualDiffusionDataset(Dataset):
    """
    Dataset for training Stage 2 Residual Conditional Diffusion.
    Loads [MRI, CT, Mask] and matches with the precomputed Out-Of-Fold coarse CT.
    Target clean residual: z_0 = (CT - Coarse_OOF) / s.
    Condition inputs: MRI, Coarse_OOF, Mask.
    """
    def __init__(self, mri_paths, ct_paths, oof_dict, residual_scale, image_size=128, is_train=True):
        self.mri_paths  = mri_paths
        self.ct_paths   = ct_paths
        self.oof_dict   = oof_dict
        self.scale      = residual_scale
        self.image_size = image_size
        self.is_train   = is_train

    def __len__(self):
        return len(self.mri_paths)

    def __getitem__(self, idx):
        mri_path = self.mri_paths[idx]
        ct_path  = self.ct_paths[idx]
        sname    = os.path.basename(mri_path)

        mri_img = Image.open(mri_path).convert('L')
        ct_img  = Image.open(ct_path).convert('L')
        
        mri_arr = np.array(mri_img, dtype=np.float32)
        ct_arr  = np.array(ct_img,  dtype=np.float32)
        
        mri_mask = derive_mri_body_mask(mri_arr)
        mri_arr  = mri_arr * mri_mask
        
        mri_res  = cv2.resize(mri_arr,  (self.image_size, self.image_size), interpolation=cv2.INTER_LINEAR)
        ct_res   = cv2.resize(ct_arr,   (self.image_size, self.image_size), interpolation=cv2.INTER_LINEAR)
        mask_res = cv2.resize(mri_mask, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)
        
        mri_norm = (mri_res / 255.0) * 2.0 - 1.0
        ct_norm  = (ct_res  / 255.0) * 2.0 - 1.0
        
        mri_norm[mask_res < 0.5] = -1.0
        ct_norm[mask_res < 0.5]  = -1.0
        
        t_mri  = torch.from_numpy(mri_norm).unsqueeze(0).float()
        t_ct   = torch.from_numpy(ct_norm).unsqueeze(0).float()
        t_mask = torch.from_numpy(mask_res).unsqueeze(0).float()
        
        # Load matching precomputed OOF coarse prediction
        t_coarse = self.oof_dict[sname] # [1, H, W]

        # Target residual: r = CT - Coarse
        r = t_ct - t_coarse
        z_0 = r / self.scale

        return {
            "mri": t_mri,
            "ct": t_ct,
            "coarse": t_coarse,
            "mask": t_mask,
            "z_0": z_0,
            "slice_name": sname,
            "patient_id": sname.split("_")[0]
        }

residual_train_dataset = SynthRADResidualDiffusionDataset(
    train_mri, train_ct, OOF_PREDICTIONS, RESIDUAL_SCALE, image_size=IMG_SIZE, is_train=True
)
residual_train_loader = DataLoader(
    residual_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)

print(f"[OK] Residual Diffusion Training Loader instantiated: {len(residual_train_loader)} batches ({len(residual_train_dataset)} slices).")


In [ ]:
# =====================================================================
# CELL 14: STAGE 2 — RESIDUAL CONDITIONAL DIFFUSION TRAINING ENGINE
# =====================================================================

# Clinical Evaluation Suite Functions (MAE, SSIM, PSNR, Provisional HU)
PROV_HU_MIN   = CONFIG["provisional_hu_mapping"]["assumed_hu_min"]
PROV_HU_MAX   = CONFIG["provisional_hu_mapping"]["assumed_hu_max"]
PROV_HU_RANGE = CONFIG["provisional_hu_mapping"]["assumed_hu_range"]
PROV_BONE_TH  = CONFIG["provisional_hu_mapping"]["bone_threshold_hu"]

def norm_to_provisional_hu(norm_arr):
    """Converts normalized [-1.0, 1.0] array to provisional Hounsfield Units [-1000, 1500]."""
    unit = np.clip((norm_arr + 1.0) / 2.0, 0.0, 1.0)
    return unit * PROV_HU_RANGE + PROV_HU_MIN

def evaluate_slice(pred_norm, gt_norm, mask_arr=None):
    """
    Computes normalized PNG metrics and provisional HU metrics on a 2D slice.
    Normalized metrics (MAE, SSIM, PSNR) are the primary basis for research evaluation.
    """
    pred_c = np.clip(pred_norm, -1.0, 1.0)
    gt_c   = np.clip(gt_norm,   -1.0, 1.0)

    pred_u = (pred_c + 1.0) / 2.0
    gt_u   = (gt_c   + 1.0) / 2.0

    mae_norm  = float(np.mean(np.abs(pred_u - gt_u)))
    rmse_norm = float(np.sqrt(np.mean((pred_u - gt_u) ** 2)))
    psnr_norm = float(compute_psnr(gt_u, pred_u, data_range=1.0))
    ssim_val, ssim_map = compute_ssim(gt_u, pred_u, data_range=1.0, full=True)
    global_ssim = float(ssim_val)

    if mask_arr is not None:
        body = mask_arr > 0.5
        if body.sum() > 0:
            masked_mae_norm = float(np.mean(np.abs(pred_u[body] - gt_u[body])))
            masked_ssim     = float(np.mean(ssim_map[body]))
            masked_mse      = float(np.mean((pred_u[body] - gt_u[body]) ** 2))
            masked_psnr     = float(10.0 * np.log10(1.0 / (masked_mse + 1e-8)))
        else:
            masked_mae_norm = mae_norm
            masked_ssim     = global_ssim
            masked_psnr     = psnr_norm
    else:
        body = np.ones_like(pred_norm, dtype=bool)
        masked_mae_norm = mae_norm
        masked_ssim     = global_ssim
        masked_psnr     = psnr_norm

    pred_hu = norm_to_provisional_hu(pred_c)
    gt_hu   = norm_to_provisional_hu(gt_c)

    hu_mae = float(np.mean(np.abs(pred_hu - gt_hu)))
    if mask_arr is not None and body.sum() > 0:
        masked_hu_mae = float(np.mean(np.abs(pred_hu[body] - gt_hu[body])))
    else:
        masked_hu_mae = hu_mae

    real_bone = (gt_hu > PROV_BONE_TH) & body
    fake_bone = (pred_hu > PROV_BONE_TH) & body
    intersection = np.sum(real_bone & fake_bone)
    total_area   = np.sum(real_bone) + np.sum(fake_bone)
    provisional_bone_dice = float((2.0 * intersection) / (total_area + 1e-8)) if total_area > 0 else 1.0

    return {
        "MAE_norm": mae_norm,
        "Masked_MAE_norm": masked_mae_norm,
        "PSNR_dB": psnr_norm,
        "Masked_PSNR_dB": masked_psnr,
        "SSIM": global_ssim,
        "Masked_SSIM": masked_ssim,
        "Provisional_HU_MAE": hu_mae,
        "Provisional_Masked_HU_MAE": masked_hu_mae,
        "Provisional_Bone_Dice": provisional_bone_dice
    }

DIFF_EPOCHS = CONFIG["stage2_residual_diffusion"]["epochs"]
DIFF_LR     = CONFIG["stage2_residual_diffusion"]["learning_rate"]
DIFF_WD     = CONFIG["stage2_residual_diffusion"]["weight_decay"]
CKPT_INT    = CONFIG["stage2_residual_diffusion"]["checkpoint_interval_epochs"]
VAL_INT     = CONFIG["stage2_residual_diffusion"]["validation_interval_epochs"]

# Deterministic validation noise pool (ensures repeatable validation synthesis across epochs)
gen_v = torch.Generator()
gen_v.manual_seed(SEED)
FIXED_VAL_NOISE_DICT = {
    i: torch.randn((1, 1, IMG_SIZE, IMG_SIZE), generator=gen_v) for i in range(len(val_dataset_eval))
}

def predict_ensemble_coarse(mri, mask):
    """Ensemble coarse CT prediction: average of all 4 fold ResUNet instances."""
    preds = []
    for k in range(1, N_FOLDS + 1):
        m_k = COARSE_MODELS[k]
        m_k.eval()
        with torch.no_grad():
            c_k = m_k(mri)
            c_k = torch.where(mask > 0.5, c_k, torch.full_like(c_k, -1.0))
            preds.append(c_k)
    return torch.stack(preds, dim=0).mean(dim=0)

def evaluate_residual_validation(model, ddim_steps=50):
    """Evaluates validation synthesis using ensemble coarse CT + residual DDIM refinement."""
    model.eval()
    slice_records = []
    patient_slices = {}
    
    global_idx = 0
    with torch.no_grad():
        for batch in val_loader:
            mri  = batch["mri"].to(device)
            ct   = batch["ct"].to(device)
            mask = batch["mask"].to(device)
            pids = batch["patient_id"]
            snames = batch["slice_name"]
            b_sz = mri.size(0)

            # 1. Ensemble coarse CT
            coarse_ens = predict_ensemble_coarse(mri, mask)

            # 2. Gather fixed noise
            b_noise = torch.cat([FIXED_VAL_NOISE_DICT[global_idx + i] for i in range(b_sz)], dim=0).to(device)
            global_idx += b_sz

            # 3. Residual DDIM reverse sampling
            sct, _ = sample_residual_ddim(
                model, mri, coarse_ens, mask, RESIDUAL_SCALE, ddim_steps=ddim_steps, initial_noise=b_noise
            )

            sct_np  = sct.squeeze(1).cpu().numpy()
            ct_np   = ct.squeeze(1).cpu().numpy()
            mask_np = mask.squeeze(1).cpu().numpy()

            for i in range(b_sz):
                pid = pids[i]
                res = evaluate_slice(sct_np[i], ct_np[i], mask_np[i])
                res["patient_id"] = pid
                res["slice_name"] = snames[i]
                slice_records.append(res)
                patient_slices.setdefault(pid, []).append(res)

    df_sl = pd.DataFrame(slice_records)
    patient_records = []
    for pid, records in patient_slices.items():
        pdf = pd.DataFrame(records)
        patient_records.append({
            "patient_id": pid,
            "Masked_MAE_norm": float(pdf["Masked_MAE_norm"].mean()),
            "Masked_SSIM": float(pdf["Masked_SSIM"].mean()),
            "Masked_PSNR_dB": float(pdf["Masked_PSNR_dB"].mean())
        })
    df_pt = pd.DataFrame(patient_records)

    p_mae  = float(df_pt["Masked_MAE_norm"].mean())
    p_ssim = float(df_pt["Masked_SSIM"].mean())
    p_psnr = float(df_pt["Masked_PSNR_dB"].mean())
    score  = p_mae + 0.10 * (1.0 - p_ssim)
    
    return score, p_mae, p_ssim, p_psnr, df_sl, df_pt

if RUN_RESIDUAL_TRAINING:
    diff_optimizer = torch.optim.AdamW(residual_diff_model.parameters(), lr=DIFF_LR, weight_decay=DIFF_WD)
    diff_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(diff_optimizer, T_max=DIFF_EPOCHS, eta_min=1e-6)
    diff_scaler = torch.amp.GradScaler('cuda', enabled=(device.type == "cuda"))

    best_diff_score = float("inf")
    best_diff_epoch = 0
    diff_train_history = []
    diff_val_history   = []

    start_diff_time = time.time()
    print("=" * 80)
    print(f"STARTING STAGE 2 RESIDUAL CONDITIONAL DIFFUSION TRAINING ({DIFF_EPOCHS} EPOCHS)")
    print(f"Condition Channels   : [z_t, MRI, Coarse_OOF, Mask] (4 channels)")
    print(f"Target Residual      : z_0 = (CT - Coarse_OOF) / {RESIDUAL_SCALE:.4f}")
    print(f"Checkpoints Saved    : Every {CKPT_INT} Epochs + Best + Final")
    print(f"Validation Frequency : Every {VAL_INT} Epochs + Epoch 1 & {DIFF_EPOCHS}")
    print("=" * 80)

    for epoch in range(1, DIFF_EPOCHS + 1):
        residual_diff_model.train()
        running_loss = 0.0
        
        pbar = tqdm(residual_train_loader, desc=f"Epoch [{epoch:02d}/{DIFF_EPOCHS}] Train Residual Diffusion")
        for batch in pbar:
            mri    = batch["mri"].to(device)
            coarse = batch["coarse"].to(device)
            mask   = batch["mask"].to(device)
            z_0    = batch["z_0"].to(device)
            b_sz   = z_0.shape[0]

            # Sample diffusion timesteps
            t = torch.randint(0, sched.timesteps, (b_sz,), device=device).long()
            z_t, noise = sched.add_noise(z_0, t)

            diff_optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
                # Condition on [z_t, MRI, Coarse, Mask]
                cond_in = torch.cat([z_t, mri, coarse, mask], dim=1)
                pred_noise = residual_diff_model(cond_in, t)
                loss = F.mse_loss(pred_noise, noise)

            diff_scaler.scale(loss).backward()
            diff_scaler.unscale_(diff_optimizer)
            torch.nn.utils.clip_grad_norm_(residual_diff_model.parameters(), max_norm=1.0)
            diff_scaler.step(diff_optimizer)
            diff_scaler.update()

            running_loss += loss.item()
            pbar.set_postfix({"Noise_MSE": f"{loss.item():.4f}"})

        diff_scheduler.step()
        ep_loss = running_loss / len(residual_train_loader)
        cur_lr = diff_optimizer.param_groups[0]["lr"]

        run_val = (epoch % VAL_INT == 0) or (epoch == 1) or (epoch == DIFF_EPOCHS)
        val_rec = {"epoch": epoch, "lr": cur_lr, "train_noise_mse": ep_loss, "val_evaluated": run_val}
        saved_flag = ""

        if run_val:
            print(f"--> Running Stage 2 Validation Synthesis for Epoch {epoch}...")
            v_score, v_mae, v_ssim, v_psnr, _, _ = evaluate_residual_validation(
                residual_diff_model, ddim_steps=CONFIG["stage2_residual_diffusion"]["ddim_inference_steps"]
            )
            val_rec.update({
                "val_score": v_score,
                "Masked_MAE_norm": v_mae,
                "Masked_SSIM": v_ssim,
                "Masked_PSNR_dB": v_psnr
            })

            if v_score < best_diff_score:
                best_diff_score = v_score
                best_diff_epoch = epoch
                best_ckpt_file = os.path.join(DIR_CKPT_DIFF, "best_model4_residual_diffusion.pth")
                torch.save({
                    "epoch": best_diff_epoch,
                    "model_state_dict": residual_diff_model.state_dict(),
                    "optimizer_state_dict": diff_optimizer.state_dict(),
                    "lr_scheduler_state_dict": diff_scheduler.state_dict(),
                    "residual_scale_s": RESIDUAL_SCALE,
                    "val_score": best_diff_score,
                    "config": CONFIG
                }, best_ckpt_file)
                saved_flag = " ⭐ [BEST STAGE 2 CHECKPOINT SAVED]"

            print(f"Epoch [{epoch:02d}/{DIFF_EPOCHS}] | Noise MSE: {ep_loss:.4f} | "
                  f"Score: {v_score:.4f} (Masked MAE: {v_mae:.4f}, Masked SSIM: {v_ssim:.4f}, PSNR: {v_psnr:.2f} dB){saved_flag}")
        else:
            print(f"Epoch [{epoch:02d}/{DIFF_EPOCHS}] | Noise MSE: {ep_loss:.4f} (LR: {cur_lr:.2e})")

        # Periodic checkpoint
        if epoch % CKPT_INT == 0:
            p_file = os.path.join(DIR_CKPT_DIFF, f"checkpoint_epoch_{epoch:03d}.pth")
            torch.save({
                "epoch": epoch,
                "model_state_dict": residual_diff_model.state_dict(),
                "optimizer_state_dict": diff_optimizer.state_dict(),
                "residual_scale_s": RESIDUAL_SCALE,
                "val_score": val_rec.get("val_score", None),
                "config": CONFIG
            }, p_file)
            print(f"  [CKPT] Periodic checkpoint saved: {p_file}")

        diff_train_history.append({"epoch": epoch, "lr": cur_lr, "train_noise_mse": ep_loss})
        diff_val_history.append(val_rec)

    # Save final checkpoint
    final_diff_file = os.path.join(DIR_CKPT_DIFF, "final_model4_residual_diffusion.pth")
    torch.save({
        "epoch": DIFF_EPOCHS,
        "model_state_dict": residual_diff_model.state_dict(),
        "residual_scale_s": RESIDUAL_SCALE,
        "config": CONFIG
    }, final_diff_file)

    pd.DataFrame(diff_train_history).to_csv(os.path.join(DIR_METRICS, "residual_diffusion_train_history.csv"), index=False)
    pd.DataFrame(diff_val_history).to_csv(os.path.join(DIR_METRICS, "residual_diffusion_val_history.csv"), index=False)
    print("=" * 80)
    print(f"STAGE 2 RESIDUAL DIFFUSION TRAINING COMPLETED in {(time.time() - start_diff_time)/60.0:.2f} minutes.")
    print(f"Best Validation Epoch : {best_diff_epoch} (Score: {best_diff_score:.4f})")
    print("=" * 80)
else:
    print("[INFO] RUN_RESIDUAL_TRAINING is set to False.")
    best_ckpt_file = os.path.join(DIR_CKPT_DIFF, "best_model4_residual_diffusion.pth")
    if os.path.exists(best_ckpt_file):
        data = torch.load(best_ckpt_file, map_location=device)
        residual_diff_model.load_state_dict(data["model_state_dict"])
        print(f"[LOADED] Pre-existing best Model 4 checkpoint from: {best_ckpt_file}")
    else:
        print("[WARNING] No trained Model 4 checkpoint found. Enable RUN_RESIDUAL_TRAINING = True.")


In [ ]:
# =====================================================================
# CELL 15: FROZEN BEST-MODEL VALIDATION & CLINICAL METRIC EXPORT
# =====================================================================

best_ckpt_file = os.path.join(DIR_CKPT_DIFF, "best_model4_residual_diffusion.pth")
if os.path.exists(best_ckpt_file):
    ckpt_d = torch.load(best_ckpt_file, map_location=device)
    residual_diff_model.load_state_dict(ckpt_d["model_state_dict"])
    residual_diff_model.eval()
    print(f"[OK] Loaded best Model 4 denoiser (Epoch {ckpt_d.get('epoch', 'N/A')}) for frozen evaluation.")
else:
    print("[NOTICE] Using current model weights for validation evaluation.")

print("")
print("Running comprehensive evaluation across FROZEN VALIDATION COHORT (6 patients, 372 slices)...")
val_score, v_mae, v_ssim, v_psnr, df_val_slices, df_val_patients = evaluate_residual_validation(
    residual_diff_model, ddim_steps=CONFIG["stage2_residual_diffusion"]["ddim_inference_steps"]
)

# Export slice and patient level CSVs
df_val_slices.to_csv(os.path.join(DIR_METRICS, "model4_val_slice_metrics.csv"), index=False)
df_val_patients.to_csv(os.path.join(DIR_METRICS, "model4_val_patient_metrics.csv"), index=False)

print("=" * 80)
print("MODEL 4 FROZEN VALIDATION RESULTS (PATIENT-LEVEL MEAN ± STD):")
print("=" * 80)
print(f"  Masked MAE (norm)         : {df_val_patients['Masked_MAE_norm'].mean():.4f} ± {df_val_patients['Masked_MAE_norm'].std():.4f}")
print(f"  Masked SSIM               : {df_val_patients['Masked_SSIM'].mean():.4f} ± {df_val_patients['Masked_SSIM'].std():.4f}")
print(f"  Masked PSNR (dB)          : {df_val_patients['Masked_PSNR_dB'].mean():.2f} ± {df_val_patients['Masked_PSNR_dB'].std():.2f}")
print(f"  Validation Composite Score: {val_score:.4f}")
print("=" * 80)


In [ ]:
# =====================================================================
# CELL 16: SCIENTIFIC FIGURES & 5-PANEL QUALITATIVE VISUALIZATIONS
# =====================================================================

# 1. Plot Training History if available
val_hist_csv = os.path.join(DIR_METRICS, "residual_diffusion_val_history.csv")
if os.path.exists(val_hist_csv):
    df_vh = pd.read_csv(val_hist_csv)
    fig, ax1 = plt.subplots(figsize=(8, 4), dpi=120)
    ax1.plot(df_vh["epoch"], df_vh["train_noise_mse"], color="tab:blue", lw=2, label="Train Noise MSE")
    ax1.set_xlabel("Epoch", fontsize=11)
    ax1.set_ylabel("Noise MSE Loss", fontsize=11)
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc="upper right")
    plt.title("Model 4: Residual Diffusion Training Dynamics", fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_FIGURES, "model4_training_curves.png"))
    plt.show()

# 2. Qualitative 5-Panel Visual Comparison:
# Columns: [MRI Input | Coarse ResUNet CT | Generated Residual | Final sCT | Absolute Error |sCT - CT|]
fig, axes = plt.subplots(2, 5, figsize=(20, 8), dpi=120)

val_sample_batch = next(iter(val_loader))
mri_b  = val_sample_batch["mri"][:2].to(device)
ct_b   = val_sample_batch["ct"][:2].to(device)
mask_b = val_sample_batch["mask"][:2].to(device)
pids_b = val_sample_batch["patient_id"][:2]

coarse_ens_b = predict_ensemble_coarse(mri_b, mask_b)
sct_b, res_b = sample_residual_ddim(
    residual_diff_model, mri_b, coarse_ens_b, mask_b, RESIDUAL_SCALE, ddim_steps=50
)

for row in range(2):
    m_img = mri_b[row, 0].cpu().numpy()
    c_img = coarse_ens_b[row, 0].cpu().numpy()
    r_img = res_b[row, 0].cpu().numpy()
    s_img = sct_b[row, 0].cpu().numpy()
    g_img = ct_b[row, 0].cpu().numpy()
    diff  = np.abs(s_img - g_img)

    axes[row, 0].imshow(m_img, cmap="gray", vmin=-1.0, vmax=1.0)
    axes[row, 0].set_title(f"Input MRI ({pids_b[row]})")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(c_img, cmap="gray", vmin=-1.0, vmax=1.0)
    axes[row, 1].set_title("Stage 1 Coarse CT")
    axes[row, 1].axis("off")

    im_r = axes[row, 2].imshow(r_img, cmap="coolwarm", vmin=-0.4, vmax=0.4)
    axes[row, 2].set_title("Generated Residual r")
    axes[row, 2].axis("off")
    fig.colorbar(im_r, ax=axes[row, 2], fraction=0.046, pad=0.04)

    axes[row, 3].imshow(s_img, cmap="gray", vmin=-1.0, vmax=1.0)
    axes[row, 3].set_title("Model 4 Final sCT")
    axes[row, 3].axis("off")

    im_d = axes[row, 4].imshow(diff, cmap="hot", vmin=0.0, vmax=0.6)
    axes[row, 4].set_title("Error |sCT - Target CT|")
    axes[row, 4].axis("off")
    fig.colorbar(im_d, ax=axes[row, 4], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(os.path.join(DIR_FIGURES, "model4_qualitative_5panel.png"))
plt.show()
print(f"[OK] Scientific qualitative visualizations saved to: {DIR_FIGURES}")


In [ ]:
# =====================================================================
# CELL 17: ABLATION STUDY FRAMEWORK & BENCHMARK COMPARISON TABLE
# =====================================================================

# Benchmark comparison table across the four models in the research project
study_benchmark = [
    {
        "Model Paradigm": "Model 1: Supervised ResUNet",
        "Role in Research": "Direct Supervised Regression Baseline",
        "Trainable Params": "33.7M",
        "Inference Mechanism": "Single-Pass Deterministic Feedforward",
        "Primary Loss": "Masked L1 + Masked SSIM",
        "Validation Masked MAE": "0.0682 (Benchmark)",
        "Validation Masked SSIM": "0.8524 (Benchmark)"
    },
    {
        "Model Paradigm": "Model 2: Paired Pix2Pix cGAN",
        "Role in Research": "Adversarial Translation Baseline",
        "Trainable Params": "41.8M (Gen: 33.7M, Disc: 8.1M)",
        "Inference Mechanism": "Single-Pass Feedforward Generator",
        "Primary Loss": "Adversarial LSGAN + 100x Masked L1",
        "Validation Masked MAE": "0.0615 (Benchmark)",
        "Validation Masked SSIM": "0.8651 (Benchmark)"
    },
    {
        "Model Paradigm": "Model 3: Standard Conditional DDPM",
        "Role in Research": "Full-Image Conditional Diffusion Baseline",
        "Trainable Params": "3.46M",
        "Inference Mechanism": "50-Step Deterministic DDIM",
        "Primary Loss": "Noise MSE(predicted, true)",
        "Validation Masked MAE": "Pending Full Retrain",
        "Validation Masked SSIM": "Pending Full Retrain"
    },
    {
        "Model Paradigm": "Model 4: Proposed Residual Diffusion",
        "Role in Research": "Cross-Fitted ResUNet + Residual Diffusion Refinement",
        "Trainable Params": "Coarse: 4x 8.4M, Diff: 3.46M",
        "Inference Mechanism": "Ensemble Coarse + 50-Step Residual DDIM",
        "Primary Loss": "ResUNet L1/SSIM + Residual Noise MSE",
        "Validation Masked MAE": f"{df_val_patients['Masked_MAE_norm'].mean():.4f}" if 'df_val_patients' in locals() else "N/A",
        "Validation Masked SSIM": f"{df_val_patients['Masked_SSIM'].mean():.4f}" if 'df_val_patients' in locals() else "N/A"
    }
]

df_bench = pd.DataFrame(study_benchmark)
df_bench.to_csv(os.path.join(DIR_METRICS, "four_models_benchmark_comparison.csv"), index=False)

print("=" * 115)
print("                         FOUR-MODEL RESEARCH BENCHMARK COMPARISON TABLE                                  ")
print("=" * 115)
print(df_bench.to_string(index=False))
print("=" * 115)

# Optional ablation configuration summary
ablations_spec = {
    "E0": "Coarse ResUNet ensemble alone (no diffusion refinement)",
    "E1": "Model 3 full-CT conditional diffusion baseline",
    "E2": "Full-CT diffusion conditioned on MRI and coarse CT",
    "E3": "Proposed Model 4 Cross-Fitted Residual Diffusion (Primary)",
    "E4": "Residual diffusion trained with in-sample coarse predictions (leakage diagnostic)",
    "E5": "Residual diffusion with auxiliary gradient/bone loss"
}
with open(os.path.join(DIR_CONFIGS, "ablations_specification.json"), "w", encoding="utf-8") as f:
    json.dump(ablations_spec, f, indent=2)


In [ ]:
# =====================================================================
# CELL 18: RESEARCH REPORT & VERIFIED DELIVERABLES ZIP PACKAGING
# =====================================================================

# 1. Generate text report
report_file = os.path.join(DIR_REPORTS, "model4_experiment_report.txt")
with open(report_file, "w", encoding="utf-8") as f:
    f.write("=" * 80 + "\n")
    f.write("SYNTHRAD2023 MODEL 4: CROSS-FITTED RESIDUAL CONDITIONAL DIFFUSION REPORT\n")
    f.write("=" * 80 + "\n\n")
    f.write(f"Experiment Name        : {CONFIG['experiment_name']}\n")
    f.write(f"Output Directory       : {BASE_OUT_DIR}\n")
    f.write(f"Residual Scale (s)     : {RESIDUAL_SCALE:.5f}\n\n")
    f.write("Validation Results (Patient-Level Mean +/- Std):\n")
    if 'df_val_patients' in locals():
        f.write(f"  - Masked MAE (norm)  : {df_val_patients['Masked_MAE_norm'].mean():.4f} +/- {df_val_patients['Masked_MAE_norm'].std():.4f}\n")
        f.write(f"  - Masked SSIM        : {df_val_patients['Masked_SSIM'].mean():.4f} +/- {df_val_patients['Masked_SSIM'].std():.4f}\n")
        f.write(f"  - Masked PSNR (dB)   : {df_val_patients['Masked_PSNR_dB'].mean():.2f} +/- {df_val_patients['Masked_PSNR_dB'].std():.2f}\n")
    f.write("\nKey Scientific Protocols:\n")
    f.write("  1. Coarse ResUNet trained with 4-fold patient-level cross-fitting.\n")
    f.write("  2. Residual diffusion trained exclusively on Out-Of-Fold (OOF) residuals.\n")
    f.write("  3. Coarse CT during inference computed via 4-fold deterministic ensemble.\n")
    f.write("  4. Held-out test patients strictly isolated.\n")
    f.write("=" * 80 + "\n")

print(f"[OK] Research report generated: {report_file}")

# 2. Package Deliverables ZIP Archive
zip_dest = os.path.join(os.path.dirname(BASE_OUT_DIR), "synthrad_model4_residual_diffusion_deliverables.zip")
if not os.path.exists(os.path.dirname(BASE_OUT_DIR)):
    zip_dest = "synthrad_model4_residual_diffusion_deliverables.zip"

print(f"Creating verified deliverables ZIP: {zip_dest}")
with zipfile.ZipFile(zip_dest, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(BASE_OUT_DIR):
        for file in sorted(files):
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, BASE_OUT_DIR)
            zipf.write(file_path, arcname)

archive_size_mb = os.path.getsize(zip_dest) / (1024 ** 2)
print(f"[OK] Deliverables ZIP archive verified: {zip_dest} ({archive_size_mb:.2f} MB)")

with zipfile.ZipFile(zip_dest, "r") as z:
    archive_members = z.namelist()

print(f"Total verified archived files: {len(archive_members)}")
for item in sorted(archive_members):
    if item.endswith(".pth") or item.endswith(".json") or item.endswith(".csv") or item.endswith(".png"):
        print(f"  - {item}")

print("=" * 80)
print("MODEL 4 PIPELINE EXECUTION AND PACKAGING COMPLETED.")
print("=" * 80)


In [ ]:
# =====================================================================
# CELL 19: OPTIONAL FROZEN TEST EVALUATION (STRICTLY GATED & DISABLED)
# =====================================================================

# SAFETY GUARD: Kept strictly False by default.
# Only enable after final checkpoint selection and experimental settings are completely frozen.
RUN_FINAL_TEST_EVALUATION = False

print("=" * 80)
print("CELL 19: FROZEN TEST SET EVALUATION (HELD-OUT 6 PATIENTS, 363 SLICES)")
print(f"RUN_FINAL_TEST_EVALUATION status: {RUN_FINAL_TEST_EVALUATION}")
print("=" * 80)

if not RUN_FINAL_TEST_EVALUATION:
    print("--> Held-out test evaluation is DISABLED.")
    print("    This prevents data leakage and preserves test set isolation during research.")
    print("    To evaluate the frozen best checkpoint on the test cohort, explicitly set:")
    print("    RUN_FINAL_TEST_EVALUATION = True and run this cell manually.")
else:
    print("Running evaluation on FROZEN HELD-OUT TEST COHORT (6 patients: 1BA054, 1BA097, 1BA172, 1BA234, 1BA256, 1BA307)...")
    test_eval_loader = DataLoader(test_dataset_eval, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    
    test_slice_records = []
    test_patient_slices = {}
    
    with torch.no_grad():
        for batch in tqdm(test_eval_loader, desc="Evaluating Test Set"):
            mri  = batch["mri"].to(device)
            ct   = batch["ct"].to(device)
            mask = batch["mask"].to(device)
            pids = batch["patient_id"]
            snames = batch["slice_name"]
            b_sz = mri.size(0)

            # Ensemble coarse CT
            coarse_ens = predict_ensemble_coarse(mri, mask)

            # Residual DDIM sampling
            sct, _ = sample_residual_ddim(
                residual_diff_model, mri, coarse_ens, mask, RESIDUAL_SCALE, ddim_steps=50
            )

            sct_np  = sct.squeeze(1).cpu().numpy()
            ct_np   = ct.squeeze(1).cpu().numpy()
            mask_np = mask.squeeze(1).cpu().numpy()

            for i in range(b_sz):
                pid = pids[i]
                res = evaluate_slice(sct_np[i], ct_np[i], mask_np[i])
                res["patient_id"] = pid
                res["slice_name"] = snames[i]
                test_slice_records.append(res)
                test_patient_slices.setdefault(pid, []).append(res)

    df_test_slices = pd.DataFrame(test_slice_records)
    test_patient_records = []
    for pid, records in test_patient_slices.items():
        pdf = pd.DataFrame(records)
        test_patient_records.append({
            "patient_id": pid,
            "Masked_MAE_norm": float(pdf["Masked_MAE_norm"].mean()),
            "Masked_SSIM": float(pdf["Masked_SSIM"].mean()),
            "Masked_PSNR_dB": float(pdf["Masked_PSNR_dB"].mean()),
            "Provisional_Masked_HU_MAE": float(pdf["Provisional_Masked_HU_MAE"].mean())
        })
    df_test_patients = pd.DataFrame(test_patient_records)

    df_test_slices.to_csv(os.path.join(DIR_METRICS, "model4_test_slice_metrics.csv"), index=False)
    df_test_patients.to_csv(os.path.join(DIR_METRICS, "model4_test_patient_metrics.csv"), index=False)

    print("")
    print("=" * 80)
    print("FINAL HELD-OUT TEST EVALUATION RESULTS (PATIENT-LEVEL MEAN ± STD):")
    print("=" * 80)
    print(f"  Test Masked MAE (norm)         : {df_test_patients['Masked_MAE_norm'].mean():.4f} ± {df_test_patients['Masked_MAE_norm'].std():.4f}")
    print(f"  Test Masked SSIM               : {df_test_patients['Masked_SSIM'].mean():.4f} ± {df_test_patients['Masked_SSIM'].std():.4f}")
    print(f"  Test Masked PSNR (dB)          : {df_test_patients['Masked_PSNR_dB'].mean():.2f} ± {df_test_patients['Masked_PSNR_dB'].std():.2f}")
    print("=" * 80)
